# 15a · 生成器闸门：能不能跑、用什么结构、评哪一批

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → 第二阶段 F ✅ → 第三阶段 G
                                 ├─ 15  前提查验 ✅（多突变池、测量压顶、泄漏面）
                                 ├─ 15a 生成器闸门  ← 你在这里
                                 ├─ 15b 生成 + 两个能量做选择
                                 ├─ 15c 固定哪些位点
                                 ├─ 15d 平台化
                                 └─ 15e 讲法
```

## 为什么要有这一章

第 15 章定了"做什么"，这一章回答"**做得了吗**"。三个问题，一个答不上就得改方案：

| 问题 | 为什么卡脖子 |
|---|---|
| 生成器在这台机器上跑不跑得起来 | ESM-IF 要编译三个 C++ 扩展；ProteinMPNN 不在 PyPI 上 |
| 考试蛋白有没有可用的三维结构 | 反向折叠要主链坐标，而我们本地只存了预抽的结构特征 |
| 设计靶标到底是哪一段、什么样的结构才算能用 | 100/108 是 domainome，测的是**结构域**不是整条链 |

**规矩**：所有筛选阈值必须在 **train split** 上定或用公开的既定分界线，**不许看 test 的结果再调**。

In [ ]:
import os
import subprocess
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)

MPNN = ROOT / "data/tools/proteinmpnn"          # config/sources.toml 钉了 commit
STRUCT = ROOT / "data/structures_test"          # AlphaFold v6，108 个考试蛋白
MODELS = ROOT / "data/models"
print("ProteinMPNN:", MPNN.exists(), " 结构:", STRUCT.exists())

---

## 第 1 步 · 生成器跑不跑得起来

### ProteinMPNN

不在 PyPI 上。它是**工具**（代码 + 权重），不是数据集，所以**不进 `config/sources.toml`**——raw 桶只收数据类型的文件（`.gz .zip .parquet .tar .tsv .csv .txt .json` 等），`.py` 和 `.pt` 会被 `raw_validator` 挡下来，这条规矩是对的。

版本就钉在这里：**GitHub commit**
`8907e667`（仓库 2023-06-27 之后再没动过）。权重取 **`v_48_020`**——这是 Sumida 2024 用的那个
checkpoint，也是 ABACUS-T 对标的那个，用同一个权重我们的数字才和文献可比。

In [ ]:
import torch

ckpt = torch.load(MPNN / "vanilla_model_weights/v_48_020.pt", map_location="cpu", weights_only=True)
print(f"num_edges   {ckpt['num_edges']}")
print(f"noise_level {ckpt['noise_level']}")
print(f"参数张量    {len(ckpt['model_state_dict'])}")

**冒烟测试**：拿一个小而折叠良好的结构域（1ENH，engrailed homeodomain，48 个残基）跑一遍。
看两件事——跑不跑得通，以及**序列恢复率在不在论文报的量级**（ProteinMPNN 约 0.50）。

In [ ]:
work = Path("/tmp/mpnn_gate")
(work / "in").mkdir(parents=True, exist_ok=True)
(work / "out").mkdir(parents=True, exist_ok=True)
(work / "in/1ENH.pdb").write_bytes((ROOT / "data/stability/1ENH.pdb").read_bytes())

cmd = [
    "python", str(MPNN / "protein_mpnn_run.py"),
    "--pdb_path", str(work / "in/1ENH.pdb"),
    "--path_to_model_weights", str(MPNN / "vanilla_model_weights"),
    "--model_name", "v_48_020",
    "--out_folder", str(work / "out"),
    "--num_seq_per_target", "4",
    "--sampling_temp", "0.1",
    "--seed", "37",
]
r = subprocess.run(cmd, capture_output=True, text=True)
print([l for l in r.stdout.splitlines() if "generated in" in l or "Number of edges" in l])
print()
print((work / "out/seqs/1ENH.fa").read_text())

**结果**：跑通，48 个残基 4 条序列约 0.25 秒（CPU），序列恢复率 **0.40–0.48**，和论文的约 0.50 一致。

顺带记一笔肉眼可见的现象：生成的序列**富含 K 和 E**（`KFSKKQKEILEKKFKENPII…`）。这是
ProteinMPNN 已知的带电表面偏好，15a 正式对照时要量化它——氨基酸组成的 KL 散度是 ABACUS-T 用的指标之一。

### ESM-IF：本地跑不了，推到 Linux/Batch

ESM-IF 的 GVP 结构需要 `torch_sparse` / `torch_scatter` / `torch_cluster`，这三个要针对精确的 torch
版本编译 C++ 扩展。本机是 **macOS ARM + torch 2.14**，没有预编译轮子。

In [ ]:
import importlib

for m in ("torch_geometric", "torch_sparse", "torch_scatter", "torch_cluster", "esm"):
    try:
        mod = importlib.import_module(m)
        print(f"  {m:<16} {getattr(mod, '__version__', '?'):<8} OK")
    except ImportError:
        print(f"  {m:<16} 缺失")

**判断：不在本地硬扛。** 三个源码编译各自都脆，而 **Linux 上这三个包有预编译轮子**——项目本来就有
AWS Batch 上的 Nextflow，ESM-IF 属于那里。

**代价要写明**：15a 的本地对照因此少一个生成器。ESM-IF 的已发表数字先当参照引用
（ProteinGym 零样本 0.42，见 ABACUS-T 的表 2E），**不能算我们自己复现的**。

本地可用的组合：**ProteinMPNN + LigandMPNN**（同作者，纯 PyTorch）+ **ESM-2 采样**（已在
`data/esm2-650m`）+ 保守性基线。

---

## 第 2 步 · 结构从哪来，缺了哪些

### 来源说明（要写清楚，因为用的不是权威源）

项目的 raw 桶里有同一个 v6 发布，但它是**一个 5.18 GB 的 tar**
（`s3://…/datasets/alphafold-human/v6/UP000005640_9606_HUMAN_v6.tar`）。tar 没法从 S3 按文件抽，
为 108 个蛋白下 5 GB 不划算。所以**按蛋白从 EBI 取，版本仍然钉死 v6**——同一个发布，同样的文件，
只是传输方式不同。

一个实操细节：EBI 的 WAF 会用 403 拒绝 curl 的默认 User-Agent，必须送一个浏览器 UA。

In [ ]:
pdbs = sorted(STRUCT.glob("AF-*-F1-model_v6.pdb"))
have = {p.name.split("-")[1] for p in pdbs}

test = pd.read_parquet(MODELS / "test_variants.parquet",
                       columns=["dataset_id", "data_kind", "uniprot_id", "uniprot_position", "plddt"])
want = set(test.uniprot_id.unique())
missing = sorted(want - have)

print(f"考试蛋白 {len(want)} 个，取到 {len(have)} 个，缺 {len(missing)} 个")
print(f"占盘 {sum(p.stat().st_size for p in pdbs) / 1e6:.0f} MB")
print()
seqs = pd.read_parquet(MODELS / "proteome_sequences.parquet")
m = seqs[seqs.uniprot_id.isin(missing)].assign(length=lambda d: d.sequence.str.len())
print(m[["uniprot_id", "length"]].sort_values("length").to_string(index=False))

### 规则 C（排除）：没有 AlphaFold 整链模型的蛋白

缺的 8 个**全部超过 2700 个残基**——那正是 AlphaFold 单模型的长度上限，超过的要拆成重叠片段。

这是一条**和结果无关**的排除：理由是数据可得性，在任何打分之前就确定了，不是看了结果才删的。

---

## 第 3 步 · 设计靶标是哪一段，什么样的结构算能用

### 规则 A（设计靶标）：每组实验被测位置的那一段

domainome 测的是**结构域**，不是整条链。先确认那一段是不是连续、是不是测全了。

In [ ]:
def targets(df):
    g = (df.groupby(["dataset_id", "uniprot_id", "data_kind"])
           .agg(lo=("uniprot_position", "min"), hi=("uniprot_position", "max"),
                n_pos=("uniprot_position", "nunique"), med_plddt=("plddt", "median"),
                n_var=("uniprot_position", "size"))
           .reset_index())
    g["span"] = g.hi - g.lo + 1
    g["coverage"] = g.n_pos / g.span
    return g


train = pd.read_parquet(MODELS / "train_variants.parquet",
                        columns=["dataset_id", "data_kind", "uniprot_id", "uniprot_position", "plddt"])
tr = targets(train)
print("=== TRAIN：每组实验的被测段 ===")
print(tr.groupby("data_kind").agg(实验组=("dataset_id", "size"), 跨度中位=("span", "median"),
                                   覆盖率中位=("coverage", "median"),
                                   plddt中位=("med_plddt", "median")).round(2).to_string())

domainome 的覆盖率中位数是 **1.00**——被测的那一段是**连续且测全**的，所以"被测跨度"就是结构域，
设计靶标的定义没有歧义。

### 规则 B（结构质量）：被测段的 pLDDT 中位数 ≥ 70

阈值**不由我们调**，用 AlphaFold 自己公布的信心分界线：70 以上算 confident（70–90 confident，
90 以上 very high，50–70 low，50 以下 very low）。先在 **train** 上看它卡掉多少——如果卡掉一大片，
说明这个阈值不合适。

In [ ]:
print("=== TRAIN：被测段 pLDDT 中位数的分布 ===")
print(tr.med_plddt.describe(percentiles=[0.05, 0.1, 0.25, 0.5]).round(1).to_string())
print()
for t in (50, 60, 70, 80):
    k = tr.med_plddt >= t
    print(f"  >= {t}: 保留 {k.sum():>4} / {len(tr)} 组 ({k.mean():.1%})")

train 上 pLDDT 中位数本身就很高（中位 90.3，第 5 百分位 77.0），阈值 70 只卡掉 **2%**。
所以这是一道**防呆的下限**，不是一个会挑数据的筛子。

### 把三条规则依次用到 TEST 上

In [ ]:
te = targets(test)
te["has_struct"] = te.uniprot_id.isin(have)

step0 = te
step_c = step0[step0.has_struct]
step_b = step_c[step_c.med_plddt >= 70]

for name, d in (("起点", step0), ("C 有整链模型", step_c), ("B pLDDT中位>=70", step_b)):
    print(f"  {name:<18} {len(d):>3} 组 / {d.uniprot_id.nunique():>3} 蛋白 / {d.n_var.sum():>7,} 变体")

print()
print("被 B 挡掉的:")
print(step_c[step_c.med_plddt < 70][["uniprot_id", "dataset_id", "span", "med_plddt"]]
      .round(1).to_string(index=False))

被挡掉的是 **P38398（BRCA1）**在 pLDDT 34–35 的无序区上的三组实验，和 Q5SW96 的 69.6。
在 pLDDT 35 的主链上做反向折叠没有意义，排除是对的——而且理由是 AlphaFold 自己的信心分级，不是我们的判断。

### 一个我自己犯过的错，记下来

查这一章时我先抽到 Q9NU63，看到**整条链**的全局 pLDDT 只有 56.22（38.7% 是 very low），
就断定考试集里有大量不可用的结构。**错了**——该看的是被测的那一段：

In [ ]:
q = te[te.uniprot_id == "Q9NU63"]
print(q[["dataset_id", "lo", "hi", "span", "n_pos", "med_plddt"]].round(1).to_string(index=False))
print()
print("整条链全局 pLDDT 56.22，但被测结构域那 22 个残基的中位数是 84.6。")
print("domainome 挑的本来就是折叠良好的结构域——这就是为什么规则 B 只卡掉 2%。")

---

## 第 4 步 · 一个差点踩进去的坑：两个 128 不是同一批

筛完之后得到 **128 组实验**。而附录 14x 用的也是"128 组实验"。
**数字一样，很容易以为是同一批，然后直接拿 14x 的数字当基线——那样 15b 的对照就全错了。**

In [ ]:
ours = set(step_b.dataset_id)
x14 = set(te[te.n_var >= 600].dataset_id)   # 14x 的筛法：每组至少 600 个突变，不看结构

print(f"15a 的集合 {len(ours)} 组   14x 的集合 {len(x14)} 组")
print(f"交集 {len(ours & x14)} 组 | 只在 15a {len(ours - x14)} | 只在 14x {len(x14 - ours)}")
print()
print("→ 两个 128 是不同的集合，数字相同纯属巧合。")
print("→ 15b 拿 14x 做对照时，必须把 14x 的基线重算到 15a 这 128 组上，或只比交集的 113 组。")

---

## 结论：闸门过了，三条规则写死

| | 规则 | 怎么定的 |
|---|---|---|
| **A** 设计靶标 | 每组实验**被测位置的那一段** | domainome 覆盖率中位 1.00，连续且测全，无歧义 |
| **B** 结构质量 | 被测段 **pLDDT 中位数 ≥ 70** | AlphaFold 公布的 confident 分界线，**不是我们调的**；train 上只卡掉 2% |
| **C** 可得性 | 必须有 **AlphaFold 整链模型** | 缺的 8 个全部 >2700 残基，是 AlphaFold 的长度上限；与结果无关 |

**15a 的评测集：128 组实验 / 97 个蛋白 / 199,173 个变体。**

### 生成器

| | 状态 |
|---|---|
| ProteinMPNN `v_48_020` | ✅ 本地可跑，commit 钉死，恢复率 0.40–0.48（论文约 0.50） |
| LigandMPNN | 计划纳入（同作者，纯 PyTorch） |
| ESM-2 采样 | ✅ 已在本地 |
| **ESM-IF** | ❌ 本地编译不了，**推到 Batch**；已发表的 0.42 只能当引用，不算我们复现 |

### 两条要带进 15b 的账

1. **15b 和 14x 比的时候，基线必须重算到同一批实验组上**，不能直接引用 14x 的数字。
2. **ESM-IF 缺席**要在结论里写明，别让读者以为我们跑了四个生成器。

## 记账

数字全部由本 notebook 从 `data/models`、`data/structures_test` 和钉死的 ProteinMPNN 重算。
**MLflow 还没记**——这一章没有训练模型，等 15b 有了第一个真正的对照再一起记。